## Exploratory Data Analysis: Escalation, SLA & CSAT Drivers in FinTech Operations

> This analysis explores operational support data to identify key drivers of escalations, SLA breaches, repeat contacts, and customer satisfaction. The goal is to uncover actionable insights that can inform process improvements, staffing decisions, and workflow optimizations in a FinTech support environment.

#### Setup & Connect to SQLite

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3 

sns.set(style="whitegrid")

# creating database connection
conn = sqlite3.connect('opsSLA.db')

#### Inspect available tables

In [8]:
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type = 'table'", conn)
tables

,name
0,agents
1,customers
2,escalation_reasons
3,sla_logs
4,tickets


#### Row counts & data sanity check

In [10]:
for table in tables['name']:
    print('-'*50, f'{table}', '-'*50)
    print('Count of records:', pd.read_sql(f"SELECT COUNT(*) AS count FROM {table}", conn)['count'].values[0])
    display(pd.read_sql(f"SELECT * FROM {table} LIMIT 5", conn))

-------------------------------------------------- agents --------------------------------------------------
Count of records: 100


,agent_id,agent_tier,region,experience_months
0,201,L1,EMEA,29
1,202,L3,APAC,80
2,203,L2,EMEA,77
3,204,L2,EMEA,41
4,205,L1,None,43


-------------------------------------------------- customers --------------------------------------------------
Count of records: 500


,customer_id,customer_segment,tenure_months,risk_flag
0,1001,Standard,68,1
1,1002,Standard,45,0
2,1003,Standard,2,1
3,1004,Standard,27,0
4,1005,Metal,36,0


-------------------------------------------------- escalation_reasons --------------------------------------------------
Count of records: 15


,escalation_reason_id,reason_category,description
0,1,Chargeback,Dispute raised for an unauthorized or incorrec...
1,2,Plan Benefits,Paid plan benefits not applied or incorrectly ...
2,3,Insurance,Delay or rejection in insurance claim processing
3,4,Shops,Cashback or merchant shop rewards missing
4,5,RevPoints,Incorrect RevPoints balance or redemption issue


-------------------------------------------------- sla_logs --------------------------------------------------
Count of records: 1200


,sla_id,ticket_id,sla_target_mins,actual_resolution_mins,sla_breached_flag,breach_reason
0,12001,9000,360,489,1,High backlog
1,12002,9001,240,156,0,None
2,12003,9002,240,186,0,None
3,12004,9003,240,184,0,None
4,12005,9004,240,628,1,Agent capacity


-------------------------------------------------- tickets --------------------------------------------------
Count of records: 1200


,ticket_id,created_at,resolution_time_mins,resolved_at,customer_id,agent_id,issue_category,escalation_flag,escalation_reason_id,payment_type,country,repeat_contact_flag,csat_score
0,9000,2025-09-01 00:00:00,489,2025-09-01 08:09:00,1038,258,CARDS,0,NaN,WALLET,ES,0,4
1,9001,2025-09-01 01:00:00,156,2025-09-01 03:36:00,1390,223,PAYMENTS,0,NaN,CARD,FR,0,3
2,9002,2025-09-01 02:00:00,186,2025-09-01 05:06:00,1429,254,PAYMENTS,1,15.0,BANK_TRANSFER,IT,0,3
3,9003,2025-09-01 03:00:00,184,2025-09-01 06:04:00,1075,280,CARDS,0,NaN,BANK_TRANSFER,DE,0,4
4,9004,2025-09-01 04:00:00,628,2025-09-01 14:28:00,1142,265,PAYMENTS,0,NaN,WALLET,UK,0,2


#### Data Quality Audit

In [12]:
tickets = pd.read_sql("SELECT * FROM tickets;", conn)
agents = pd.read_sql("SELECT * FROM agents;", conn)
customers = pd.read_sql("SELECT * FROM customers;", conn)
escalation_reasons = pd.read_sql("SELECT * FROM escalation_reasons;", conn)
sla_logs = pd.read_sql("SELECT * FROM sla_logs;", conn)

In [13]:
def data_quality_report(df):
    return pd.DataFrame({
        'dtype': df.dtypes,
        'missing_pct': df.isna().mean().round(3) * 100,
        'unique_values': df.nunique()
    })

data_quality_report(tickets)

,dtype,missing_pct,unique_values
ticket_id,int64,0.0,1200
created_at,object,0.0,1200
resolution_time_mins,int64,0.0,664
resolved_at,object,0.0,1187
customer_id,int64,0.0,452
agent_id,int64,0.0,100
issue_category,object,0.0,5
escalation_flag,int64,0.0,2
escalation_reason_id,float64,74.9,15
payment_type,object,0.0,3


In [14]:
data_quality_report(agents)

,dtype,missing_pct,unique_values
agent_id,int64,0.0,100
agent_tier,object,0.0,3
region,object,18.0,2
experience_months,int64,0.0,63


In [15]:
data_quality_report(sla_logs)

,dtype,missing_pct,unique_values
sla_id,int64,0.0,1200
ticket_id,int64,0.0,1200
sla_target_mins,int64,0.0,3
actual_resolution_mins,int64,0.0,664
sla_breached_flag,int64,0.0,2
breach_reason,object,52.1,3


> Initial profiling highlighted missing escalation reasons, inconsistent categorical values, and skewed resolution time distributions requiring cleanup prior to analysis.

#### Handling Missing Values

In [18]:
# Escalation reason should only exist if escalated
tickets.loc[
    (tickets['escalation_flag'] == 0),
    'escalation_reason_id'
] = None

In [40]:
# Impute CSAT using median per issue category
tickets['csat_score'] = (
    tickets
    .groupby('issue_category')['csat_score']
    .transform(lambda x: x.fillna(x.median()))
)

#### Data Cleaning (Standardization & Consistency)

Check for trailing spaces, capitilization, mixed country codes

In [43]:
tickets['payment_type'].unique()

array(['WALLET', 'CARD', 'BANK_TRANSFER'], dtype=object)

In [48]:
tickets['payment_type'].str.lower().unique()

array(['wallet', 'card', 'bank_transfer'], dtype=object)

In [25]:
tickets['issue_category'] = (
    tickets['issue_category']
    .str.strip()
    .str.title()
)

tickets['country'] = tickets['country'].str.upper()

In [50]:
tickets['issue_category'].unique()

array(['Cards', 'Payments', 'Accounts', 'Compliance', 'Credit'],
      dtype=object)

In [100]:
escalation_reasons['description'].unique()

array(['Dispute raised for an unauthorized or incorrect card charge',
       'Paid plan benefits not applied or incorrectly configured',
       'Delay or rejection in insurance claim processing',
       'Cashback or merchant shop rewards missing',
       'Incorrect RevPoints balance or redemption issue',
       'Savings account interest or withdrawal discrepancy',
       'Customer unable to update registered address details',
       'eSIM activation, setup, or connectivity problem',
       'Hotel or stay booking issue via platform',
       'Credit limit, repayment, or interest calculation issue',
       'Card payment declined despite sufficient balance',
       'Delay or failure in crypto transfer confirmation',
       'Escalated dispute with merchant over transaction',
       'Delay in customer KYC document verification',
       'Temporary restriction placed on customer account'], dtype=object)

In [102]:
customers['customer_segment'].unique()

array(['Standard', 'Metal', 'Plus', 'Premium', 'Ultra'], dtype=object)

In [110]:
escalation_reasons['reason_category'].unique()

array(['Chargeback', 'Plan Benefits', 'Insurance', 'Shops', 'RevPoints',
       'Savings', 'Account - Address Change', 'eSIM', 'Stays', 'Credit',
       'Card Payments', 'Crypto Transfers', 'Merchant Dispute',
       'KYC Verification', 'Account Restriction'], dtype=object)

In [118]:
agents['region'].isna().sum()

18

In [128]:
agents['region'].unique()

array(['EMEA', 'APAC', None], dtype=object)

In [130]:
agents.mask(agents.eq('None')).dropna()

,agent_id,agent_tier,region,experience_months
0,201,L1,EMEA,29
1,202,L3,APAC,80
2,203,L2,EMEA,77
3,204,L2,EMEA,41
5,206,L1,EMEA,89
...,...,...,...,...
93,294,L2,APAC,29
94,295,L2,EMEA,18
95,296,L1,EMEA,65
96,297,L1,APAC,12


In [136]:
agents['region'].isna().sum()

18

In [144]:
escalation_reasons['escalation_reason_id'].isna().sum()

0

> Standardized categorical fields to prevent artificial category fragmentation during aggregation.

In [22]:
tickets.columns

Index(['ticket_id', 'created_at', 'resolution_time_mins', 'resolved_at',
       'customer_id', 'agent_id', 'issue_category', 'escalation_flag',
       'escalation_reason_id', 'payment_type', 'country',
       'repeat_contact_flag', 'csat_score'],
      dtype='object')

#### Data Validation rules

- Resolution time must be ≥ 0
- Escalated tickets must have escalation reason
- SLA resolution ≥ ticket resolution time

In [65]:
# invalid resolution times
invalid_resolution = tickets[tickets['resolution_time_mins'] < 0]


In [83]:
# Escalated tickets missing reason
invalid_escalations = tickets[
    (tickets['escalation_flag'] == True) &
    (tickets['escalation_reason_id'].isna())
]
invalid_escalations

,ticket_id,created_at,resolution_time_mins,resolved_at,customer_id,agent_id,issue_category,escalation_flag,escalation_reason_id,payment_type,country,repeat_contact_flag,csat_score


In [93]:
# SLA consistency check
invalid_sla = sla_logs[
    sla_logs['actual_resolution_mins'] < tickets['resolution_time_mins']
]

> Applied business validation rules to detect inconsistent records prior to KPI computation.

**Validation: resolution must be after creation**

In [225]:
invalid_dates = tickets[tickets['resolved_at'] < tickets['created_at']]
invalid_dates.shape

(0, 13)

> Validated ticket lifecycle timestamps to ensure resolution events do not precede creation times.

#### Data Enrichment (Joining Dimensions)

In [177]:
tickets_enriched = (
    tickets
    .merge(customers, on='customer_id', how='left')
    .merge(agents, on='agent_id', how='left')
    .merge(escalation_reasons, on='escalation_reason_id', how='left')
    .merge(sla_logs, on='ticket_id', how='left')
)
tickets_enriched

,ticket_id,created_at,resolution_time_mins,resolved_at,customer_id,agent_id,issue_category,escalation_flag,escalation_reason_id,payment_type,...,agent_tier,region,experience_months,reason_category,description,sla_id,sla_target_mins,actual_resolution_mins,sla_breached_flag,breach_reason
0,9000,2025-09-01 00:00:00,489,2025-09-01 08:09:00,1038,258,Cards,0,NaN,WALLET,...,L1,APAC,42,NaN,NaN,12001,360,489,1,High backlog
1,9001,2025-09-01 01:00:00,156,2025-09-01 03:36:00,1390,223,Payments,0,NaN,CARD,...,L1,EMEA,25,NaN,NaN,12002,240,156,0,None
2,9002,2025-09-01 02:00:00,186,2025-09-01 05:06:00,1429,254,Payments,1,15.0,BANK_TRANSFER,...,L3,EMEA,49,Account Restriction,Temporary restriction placed on customer account,12003,240,186,0,None
3,9003,2025-09-01 03:00:00,184,2025-09-01 06:04:00,1075,280,Cards,0,NaN,BANK_TRANSFER,...,L1,EMEA,71,NaN,NaN,12004,240,184,0,None
4,9004,2025-09-01 04:00:00,628,2025-09-01 14:28:00,1142,265,Payments,0,NaN,WALLET,...,L1,EMEA,37,NaN,NaN,12005,240,628,1,Agent capacity
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,10195,2025-10-20 19:00:00,95,2025-10-20 20:35:00,1403,226,Compliance,1,4.0,BANK_TRANSFER,...,L2,EMEA,72,Shops,Cashback or merchant shop rewards missing,13196,720,95,0,None
1196,10196,2025-10-20 20:00:00,502,2025-10-21 04:22:00,1210,216,Credit,0,NaN,CARD,...,L1,None,46,NaN,NaN,13197,720,502,0,None
1197,10197,2025-10-20 21:00:00,275,2025-10-21 01:35:00,1229,277,Cards,1,13.0,BANK_TRANSFER,...,L2,APAC,52,Merchant Dispute,Escalated dispute with merchant over transaction,13198,720,275,0,None
1198,10198,2025-10-20 22:00:00,217,2025-10-21 01:37:00,1350,239,Payments,1,11.0,WALLET,...,L2,EMEA,32,Card Payments,Card payment declined despite sufficient balance,13199,360,217,0,None


#### Date Parsing & Validation

In [235]:
tickets_enriched['created_at'] = pd.to_datetime(tickets_enriched['created_at'])
tickets_enriched['resolved_at'] = pd.to_datetime(tickets_enriched['resolved_at'])

#### Feature Engineering

##### Feature 1: SLA Risk Flag

In [179]:
tickets_enriched['sla_risk_flag'] = (
    tickets_enriched['resolution_time_mins'] >
    tickets_enriched['sla_target_mins'] * 0.8
)
tickets_enriched['sla_risk_flag']

0        True
1       False
2       False
3       False
4        True
        ...  
1195    False
1196    False
1197    False
1198    False
1199    False
Name: sla_risk_flag, Length: 1200, dtype: bool

##### Feature 2: Resolution Speed Bucket

In [186]:
tickets_enriched['resolution_speed'] = pd.cut(
    tickets_enriched['resolution_time_mins'],
    bins=[0,120,360,720,np.inf],
    labels=['Fast','Moderate','Slow','Very Slow']
)

##### Feature 3: Escalation Severity Score

In [193]:
tickets_enriched['severity_score'] = (
    tickets_enriched['escalation_flag'].astype(int) * 2 +
    tickets_enriched['sla_breached_flag'].astype(int) * 2 +
    tickets_enriched['repeat_contact_flag'].astype(int)
)

##### Feature Engineering from Dates

1. Resoltuion Duration

In [237]:
tickets_enriched['resolution_time_hours'] = (
    (tickets_enriched['resolved_at'] - tickets_enriched['created_at'])
    .dt.total_seconds() / 3600
)

2. Time-based features

In [282]:
tickets_enriched['created_date'] = tickets_enriched['created_at'].dt.date
tickets_enriched['created_week'] = tickets_enriched['created_at'].dt.to_period('W').astype(str)
tickets_enriched['created_month'] = tickets_enriched['created_at'].dt.to_period('M').astype(str)
tickets_enriched['created_hour'] = tickets_enriched['created_at'].dt.hour
tickets_enriched['created_dow'] = tickets_enriched['created_at'].dt.day_name()

> Engineered operational risk features to support escalation prioritization and SLA monitoring.
> And, engineered time-based features to support weekly, monthly, and intraday trend analysis.

#### Outlier Detection

In [199]:
q1 = tickets_enriched['resolution_time_mins'].quantile(0.25)
q3 = tickets_enriched['resolution_time_mins'].quantile(0.75)
iqr = q3 - q1

outliers = tickets_enriched[
    tickets_enriched['resolution_time_mins'] > (q3 + 1.5 * iqr)
]


> Identified extreme resolution-time outliers to assess process bottlenecks rather than blindly removing them.

#### Pre- vs Post-Cleaning Validation

In [247]:
before = tickets.shape[1]
after = tickets_enriched.dropna().shape[1]

before,after

(13, 35)

In [278]:
tickets_enriched

,ticket_id,created_at,resolution_time_mins,resolved_at,customer_id,agent_id,issue_category,escalation_flag,escalation_reason_id,payment_type,...,breach_reason,sla_risk_flag,resolution_speed,severity_score,resolution_time_hours,created_date,created_week,created_month,created_hour,created_dow
0,9000,2025-09-01 00:00:00,489,2025-09-01 08:09:00,1038,258,Cards,0,NaN,WALLET,...,High backlog,True,Slow,2,8.150000,2025-09-01,2025-09-01/2025-09-07,2025-09,0,Monday
1,9001,2025-09-01 01:00:00,156,2025-09-01 03:36:00,1390,223,Payments,0,NaN,CARD,...,None,False,Moderate,0,2.600000,2025-09-01,2025-09-01/2025-09-07,2025-09,1,Monday
2,9002,2025-09-01 02:00:00,186,2025-09-01 05:06:00,1429,254,Payments,1,15.0,BANK_TRANSFER,...,None,False,Moderate,2,3.100000,2025-09-01,2025-09-01/2025-09-07,2025-09,2,Monday
3,9003,2025-09-01 03:00:00,184,2025-09-01 06:04:00,1075,280,Cards,0,NaN,BANK_TRANSFER,...,None,False,Moderate,0,3.066667,2025-09-01,2025-09-01/2025-09-07,2025-09,3,Monday
4,9004,2025-09-01 04:00:00,628,2025-09-01 14:28:00,1142,265,Payments,0,NaN,WALLET,...,Agent capacity,True,Slow,2,10.466667,2025-09-01,2025-09-01/2025-09-07,2025-09,4,Monday
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,10195,2025-10-20 19:00:00,95,2025-10-20 20:35:00,1403,226,Compliance,1,4.0,BANK_TRANSFER,...,None,False,Fast,3,1.583333,2025-10-20,2025-10-20/2025-10-26,2025-10,19,Monday
1196,10196,2025-10-20 20:00:00,502,2025-10-21 04:22:00,1210,216,Credit,0,NaN,CARD,...,None,False,Slow,0,8.366667,2025-10-20,2025-10-20/2025-10-26,2025-10,20,Monday
1197,10197,2025-10-20 21:00:00,275,2025-10-21 01:35:00,1229,277,Cards,1,13.0,BANK_TRANSFER,...,None,False,Moderate,2,4.583333,2025-10-20,2025-10-20/2025-10-26,2025-10,21,Monday
1198,10198,2025-10-20 22:00:00,217,2025-10-21 01:37:00,1350,239,Payments,1,11.0,WALLET,...,None,False,Moderate,2,3.616667,2025-10-20,2025-10-20/2025-10-26,2025-10,22,Monday


> **Data preparation steps improved completeness and consistency while preserving analytical integrity.**

#### **Saving all the columns from final table including these newly featured columns in the database**

In [284]:
tickets_enriched.dtypes

ticket_id                          int64
created_at                datetime64[ns]
resolution_time_mins               int64
resolved_at               datetime64[ns]
customer_id                        int64
agent_id                           int64
issue_category                    object
escalation_flag                    int64
escalation_reason_id             float64
payment_type                      object
country                           object
repeat_contact_flag                int64
csat_score                         int64
customer_segment                  object
tenure_months                      int64
risk_flag                          int64
agent_tier                        object
region                            object
experience_months                  int64
reason_category                   object
description                       object
sla_id                             int64
sla_target_mins                    int64
actual_resolution_mins             int64
sla_breached_fla

In [286]:
# creating empty table
cursor = conn.cursor()

In [290]:
cursor.execute("""
    DROP TABLE IF EXISTS tickets_esc_summary;
""")


cursor.execute("""CREATE TABLE tickets_esc_summary(
    ticket_id INT PRIMARY KEY,
    
    created_at datetime,
    created_date VARCHAR(50),
    resolved_at datetime,

    resolution_time_mins INT,
    resolution_time_hours float,
    
    customer_id INT,
    customer_segment VARCHAR(50),
    tenure_months INT,
    risk_flag INT,
    
    agent_id INT,
    agent_tier VARCHAR(50),
    region VARCHAR(50),
	experience_months INT,
    
    issue_category VARCHAR(50),
    escalation_flag INT,
    escalation_reason_id float,
    reason_category VARCHAR(50),
	description VARCHAR(100),
    
    payment_type VARCHAR(50), 
    country VARCHAR(50),
    repeat_contact_flag INT,
    csat_score INT,
    
	sla_id INT,
	sla_target_mins INT,
	actual_resolution_mins INT,
	sla_breached_flag INT,
	breach_reason VARCHAR(50),
	sla_risk_flag bool,
    
	
	created_week VARCHAR(50),
	created_month VARCHAR(50),
	created_hour INT,
	created_dow VARCHAR(50),

    resolution_speed VARCHAR(50),
	severity_score INT
);""")

In [294]:
tickets_enriched.to_sql('tickets_esc_summary', conn, if_exists= 'replace', index= False)

1200

In [296]:
pd.read_sql_query("""SELECT * FROM tickets_esc_summary""", conn)

,ticket_id,created_at,resolution_time_mins,resolved_at,customer_id,agent_id,issue_category,escalation_flag,escalation_reason_id,payment_type,...,breach_reason,sla_risk_flag,resolution_speed,severity_score,resolution_time_hours,created_date,created_week,created_month,created_hour,created_dow
0,9000,2025-09-01 00:00:00,489,2025-09-01 08:09:00,1038,258,Cards,0,NaN,WALLET,...,High backlog,1,Slow,2,8.150000,2025-09-01,2025-09-01/2025-09-07,2025-09,0,Monday
1,9001,2025-09-01 01:00:00,156,2025-09-01 03:36:00,1390,223,Payments,0,NaN,CARD,...,None,0,Moderate,0,2.600000,2025-09-01,2025-09-01/2025-09-07,2025-09,1,Monday
2,9002,2025-09-01 02:00:00,186,2025-09-01 05:06:00,1429,254,Payments,1,15.0,BANK_TRANSFER,...,None,0,Moderate,2,3.100000,2025-09-01,2025-09-01/2025-09-07,2025-09,2,Monday
3,9003,2025-09-01 03:00:00,184,2025-09-01 06:04:00,1075,280,Cards,0,NaN,BANK_TRANSFER,...,None,0,Moderate,0,3.066667,2025-09-01,2025-09-01/2025-09-07,2025-09,3,Monday
4,9004,2025-09-01 04:00:00,628,2025-09-01 14:28:00,1142,265,Payments,0,NaN,WALLET,...,Agent capacity,1,Slow,2,10.466667,2025-09-01,2025-09-01/2025-09-07,2025-09,4,Monday
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,10195,2025-10-20 19:00:00,95,2025-10-20 20:35:00,1403,226,Compliance,1,4.0,BANK_TRANSFER,...,None,0,Fast,3,1.583333,2025-10-20,2025-10-20/2025-10-26,2025-10,19,Monday
1196,10196,2025-10-20 20:00:00,502,2025-10-21 04:22:00,1210,216,Credit,0,NaN,CARD,...,None,0,Slow,0,8.366667,2025-10-20,2025-10-20/2025-10-26,2025-10,20,Monday
1197,10197,2025-10-20 21:00:00,275,2025-10-21 01:35:00,1229,277,Cards,1,13.0,BANK_TRANSFER,...,None,0,Moderate,2,4.583333,2025-10-20,2025-10-20/2025-10-26,2025-10,21,Monday
1198,10198,2025-10-20 22:00:00,217,2025-10-21 01:37:00,1350,239,Payments,1,11.0,WALLET,...,None,0,Moderate,2,3.616667,2025-10-20,2025-10-20/2025-10-26,2025-10,22,Monday
